In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

base = "/content/drive/MyDrive/svg_competition"

train_df = pd.read_csv(f"{base}/train.csv")
test_df = pd.read_csv(f"{base}/test.csv")
sample_sub = pd.read_csv(f"{base}/sample_submission.csv")

print(train_df.shape, test_df.shape, sample_sub.shape)

In [ ]:
!pip uninstall -y huggingface_hub transformers tokenizers hf_transfer unsloth unsloth_zoo -q

!pip install -q \
  huggingface_hub==0.35.3 \
  transformers==4.56.2 \
  tokenizers==0.22.2 \
  hf_transfer

!pip install -q unsloth==2026.3.17 unsloth_zoo==2026.3.6

In [ ]:
import os
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"

import unsloth
from unsloth import FastLanguageModel

import re
import time
import random
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import torch
import huggingface_hub
import transformers

from datasets import Dataset

# ====== SEED ======
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# ====== INFO ======
print(f"huggingface_hub: {huggingface_hub.__version__}")
print(f"transformers: {transformers.__version__}")
print(f"Torch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
CONFIG = {
    "model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit",

    "max_seq_length": 2048,
    "max_svg_len": 1000,

    "lora_r": 16,
    "lora_alpha": 16,
    "lora_dropout": 0.0,

    "learning_rate": 5e-5,
    "num_train_epochs": 3,

    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 8,

    "warmup_steps": 50,
    "weight_decay": 0.01,

    "logging_steps": 20,
    "eval_steps": 50,
    "save_steps": 100,

    "train_sample_size": 5000,
    "eval_size": 0.05,

    "max_new_tokens": 800,
    "seed": 42,

    "output_dir": "/content/drive/MyDrive/svg_competition/qwen15b_svg_lora_final3",
    }
CONFIG

In [ ]:
def clean_official_train_df(df, max_svg_len):
    df = df.dropna(subset=["prompt", "svg"]).copy()

    df["prompt"] = df["prompt"].astype(str).str.strip()
    df["svg"] = df["svg"].astype(str).str.strip()

    df = df[
        (df["prompt"].str.len() > 0) &
        (df["svg"].str.len() > 0) &
        (df["svg"].str.startswith("<svg")) &
        (df["svg"].str.contains("</svg>", regex=False)) &
        (df["svg"].str.len() <= max_svg_len)
    ].copy()

    return df


SYSTEM_PROMPT = "Return only a complete valid SVG."


def format_example(prompt, svg):
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{svg}\n"
        "<|im_end|>"
    )



train_df_clean = clean_official_train_df(
    train_df,
    max_svg_len=CONFIG["max_svg_len"]
)
print("After cleaning:", train_df_clean.shape)

print("\nSVG length summary:")
print(train_df_clean["svg"].str.len().describe())

print("\nPrompt length summary:")
print(train_df_clean["prompt"].str.len().describe())


train_df_sampled = train_df_clean.sample(
    n=min(CONFIG["train_sample_size"], len(train_df_clean)),
    random_state=CONFIG["seed"]
).reset_index(drop=True)

print("Training subset:", train_df_sampled.shape)


eval_n = max(1, int(len(train_df_sampled) * CONFIG["eval_size"]))
rng = np.random.RandomState(CONFIG["seed"])
eval_indices = rng.choice(len(train_df_sampled), size=eval_n, replace=False)

eval_df = train_df_sampled.iloc[eval_indices].reset_index(drop=True)
train_sub_df = train_df_sampled.drop(index=eval_indices).reset_index(drop=True)

print("Train split:", train_sub_df.shape)
print("Eval split:", eval_df.shape)


train_sub_df["text"] = train_sub_df.apply(
    lambda x: format_example(x["prompt"], x["svg"]),
    axis=1
)

eval_df["text"] = eval_df.apply(
    lambda x: format_example(x["prompt"], x["svg"]),
    axis=1
)

debug_train_df = train_sub_df[["prompt", "svg", "text"]].copy()
debug_eval_df = eval_df[["prompt", "svg", "text"]].copy()

from datasets import Dataset

train_dataset = Dataset.from_pandas(
    train_sub_df[["text"]],
    preserve_index=False
)

eval_dataset = Dataset.from_pandas(
    eval_df[["text"]],
    preserve_index=False
)

print(train_dataset)
print(eval_dataset)


In [ ]:
from datasets import Dataset
import re

SYSTEM_PROMPT = (
    "Return only a complete valid SVG. "
    "Output must start with <svg and end with </svg>. "
    "Do not include any explanation or markdown."
)

def clean_svg_for_training(svg: str) -> str:
    svg = str(svg).strip()


    svg = re.sub(r'\s+filling="[^"]*"', '', svg)


    svg = re.sub(r'\s{2,}', ' ', svg)

    return svg


train_sub_df = train_sub_df.copy()
eval_df = eval_df.copy()

train_sub_df["svg"] = train_sub_df["svg"].apply(clean_svg_for_training)
eval_df["svg"] = eval_df["svg"].apply(clean_svg_for_training)


train_dataset = Dataset.from_pandas(
    train_sub_df[["prompt", "svg"]],
    preserve_index=False
)

eval_dataset = Dataset.from_pandas(
    eval_df[["prompt", "svg"]],
    preserve_index=False
)

print(train_dataset)
print(eval_dataset)

def format_sft_text(example):
    prompt = str(example["prompt"]).strip()
    svg = str(example["svg"]).strip()

    text = (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{svg}\n"
        "<|im_end|>"
    )
    return {"text": text}


train_dataset = train_dataset.map(
    format_sft_text,
    remove_columns=train_dataset.column_names,
    desc="Formatting train dataset"
)

eval_dataset = eval_dataset.map(
    format_sft_text,
    remove_columns=eval_dataset.column_names,
    desc="Formatting eval dataset"
)


MAX_TEXT_CHARS = 5000

train_dataset = train_dataset.filter(
    lambda x: 0 < len(x["text"]) <= MAX_TEXT_CHARS,
    desc="Filtering long train samples"
)

eval_dataset = eval_dataset.filter(
    lambda x: 0 < len(x["text"]) <= MAX_TEXT_CHARS,
    desc="Filtering long eval samples"
)


print(train_dataset)
print(eval_dataset)

print("\n===== Sample training text preview =====\n")
print(train_dataset[0]["text"][:800])

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=CONFIG["model_name"],
    max_seq_length=CONFIG["max_seq_length"],
    dtype=None,
    load_in_4bit=True,
)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

# =========================
# 2. LoRA
# =========================
model = FastLanguageModel.get_peft_model(
    model,
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=CONFIG["seed"],
)

# =========================
# 3. print the parameters
# =========================
model.print_trainable_parameters()

print("Model loaded:", CONFIG["model_name"])
print("Max seq length:", CONFIG["max_seq_length"])
print("Tokenizer pad token:", tokenizer.pad_token)
print("Tokenizer eos token:", tokenizer.eos_token)
print("Tokenizer padding side:", tokenizer.padding_side)

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer
import torch

training_args = TrainingArguments(
    output_dir=CONFIG["output_dir"],
    num_train_epochs=CONFIG["num_train_epochs"],
    per_device_train_batch_size=CONFIG["per_device_train_batch_size"],
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=CONFIG["gradient_accumulation_steps"],
    learning_rate=CONFIG["learning_rate"],
    warmup_steps=CONFIG["warmup_steps"],
    weight_decay=CONFIG["weight_decay"],
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=CONFIG["logging_steps"],
    eval_strategy="steps",
    eval_steps=CONFIG["eval_steps"],
    save_strategy="steps",
    save_steps=CONFIG["save_steps"],
    save_total_limit=2,
    report_to="none",
    log_level="info",

    seed=CONFIG["seed"],
    remove_unused_columns=False,
    dataloader_num_workers=2,
    load_best_model_at_end=False,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    dataset_text_field="text",
    max_seq_length=CONFIG["max_seq_length"],
    packing=False,
    args=training_args,
)

train_result = trainer.train()

print("\n===== Training finished =====")
print(train_result)

if hasattr(train_result, "metrics"):
    print("\n===== Training metrics =====")
    for k, v in train_result.metrics.items():
        print(f"{k}: {v}")

In [ ]:
import os
import json

save_dir = CONFIG["output_dir"]
os.makedirs(save_dir, exist_ok=True)

model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)

with open(os.path.join(save_dir, "train_config.json"), "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Saved LoRA adapter + tokenizer to: {save_dir}")

In [ ]:
import re
import time
import torch
import pandas as pd
import xml.etree.ElementTree as ET
from tqdm.auto import tqdm
from transformers import StoppingCriteria, StoppingCriteriaList

FastLanguageModel.for_inference(model)
model.eval()
tokenizer.padding_side = "left"

model.generation_config.do_sample = False

SYSTEM_PROMPT = "Return only a complete valid SVG."

SVG_REGEX = re.compile(r"<svg[\s\S]*?</svg>", re.IGNORECASE)

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" width="256" height="256" viewBox="0 0 256 256">'
    '<rect width="256" height="256" fill="white"/>'
    '</svg>'
)

STOP_STR = "</svg>"
STOP_TOKEN_IDS = tokenizer.encode(STOP_STR, add_special_tokens=False)


class StopOnSvgEndTokens(StoppingCriteria):
    def __init__(self, stop_token_ids):
        self.stop_token_ids = stop_token_ids

    def __call__(self, input_ids, scores, **kwargs):
        if input_ids.shape[1] < len(self.stop_token_ids):
            return False
        tail = input_ids[0, -len(self.stop_token_ids):].tolist()
        return tail == self.stop_token_ids


def build_inference_text(prompt):
    return (
        "<|im_start|>system\n"
        f"{SYSTEM_PROMPT}\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{prompt.strip()}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
    )


def get_assistant_text(decoded_text):
    marker = "<|im_start|>assistant\n"
    if marker in decoded_text:
        decoded_text = decoded_text.split(marker, 1)[1]
    return decoded_text.split("<|im_end|>", 1)[0].strip()


def extract_svg(text):
    m = SVG_REGEX.search(text)
    return m.group(0).strip() if m else ""


def is_valid_svg(svg):
    if not svg:
        return False
    try:
        root = ET.fromstring(svg)
        return root.tag.endswith("svg")
    except:
        return False


def sanitize_svg(svg, max_len=5000):
    if not svg:
        return ""

    svg = svg.replace("```svg", "").replace("```", "").strip()

    m = SVG_REGEX.search(svg)
    if m:
        svg = m.group(0)

    banned = ["<script", "onload=", "onclick=", "href=", "javascript:"]
    low = svg.lower()
    if any(b in low for b in banned):
        return ""

    if re.search(r'fill="\s*"', svg):
        return ""

    if len(svg) > max_len:
        return ""

    return svg.strip()


def fallback_svg():
    return FALLBACK_SVG


def is_fallback_svg(svg):
    return svg.strip() == FALLBACK_SVG.strip()


def postprocess(decoded):
    text = get_assistant_text(decoded)
    svg = extract_svg(text)
    svg = sanitize_svg(svg)

    if not is_valid_svg(svg):
        return fallback_svg()

    return svg


def generate_svg_batch(prompts, batch_size=16, max_new_tokens=600):

    results = []
    stopping_criteria = StoppingCriteriaList([
        StopOnSvgEndTokens(STOP_TOKEN_IDS)
    ])

    total_batches = (len(prompts) + batch_size - 1) // batch_size

    for start in tqdm(
        range(0, len(prompts), batch_size),
        total=total_batches,
        desc="Generating SVGs"
    ):
        batch_prompts = prompts[start:start + batch_size]
        texts = [build_inference_text(p) for p in batch_prompts]

        inputs = tokenizer(
            texts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=2048,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05,
                stopping_criteria=stopping_criteria,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True,
            )

        decoded_batch = tokenizer.batch_decode(outputs, skip_special_tokens=False)

        for d in decoded_batch:
            results.append(postprocess(d))

    return results


# =========================
# Submission
# =========================
base = "/content/drive/MyDrive/svg_competition"

test_df = pd.read_csv(f"{base}/test.csv")

prompts = test_df["prompt"].tolist()
ids = test_df["id"].tolist()

print(f"Total samples: {len(prompts)}")

t0 = time.time()

batch_size = 16

generated_svgs = generate_svg_batch(
    prompts,
    batch_size=batch_size,
    max_new_tokens=800
)


rows = []
invalid = 0
fallback_cnt = 0

for i, svg in enumerate(generated_svgs):

    if not is_valid_svg(svg):
        invalid += 1
        svg = fallback_svg()

    if is_fallback_svg(svg):
        fallback_cnt += 1

    rows.append({
        "id": ids[i],
        "svg": svg
    })

sub_df = pd.DataFrame(rows)
sub_path = f"{base}/submission_final3.csv"
sub_df.to_csv(sub_path, index=False)

elapsed = (time.time() - t0) / 60

print("\n===== DONE =====")
print(f"Saved: {sub_path}")
print(f"Rows: {len(sub_df)}")
print(f"Invalid: {invalid}")
print(f"Fallback: {fallback_cnt}")
print(f"Fallback ratio: {fallback_cnt/len(sub_df):.3f}")
print(f"Time: {elapsed:.2f} min")

sub_df.head()